# Feline Skin Disease Detection - CNN Training\n\nRun all cells in order. Make sure you have a **GPU runtime** enabled:\n**Runtime → Change runtime type → GPU**

## 1. Verify GPU

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs available: {len(gpus)}")
for gpu in gpus:
    print(" ", gpu)

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b update/shared-cnn-classifier-head https://github.com/Data-Anish/feline-skin-disease-detection.git /content/repo
%cd /content/repo

# Symlink Drive data into the repo so the default paths work
!ln -s /content/drive/MyDrive/feline-skin-disease-detection/new_data /content/repo/new_data

## 3. Verify Data

In [ ]:
import os

for split in ["train", "val", "test"]:
    path = os.path.join("new_data", split)
    if os.path.exists(path):
        classes = sorted(os.listdir(path))
        total = sum(len(os.listdir(os.path.join(path, c))) for c in classes)
        print(f"{split}: {len(classes)} classes, {total} images - {classes}")
    else:
        print(f"WARNING: {path} not found!")

## 4. Train (5 seeds x 10 epochs)

In [ ]:
import sys
import numpy as np

sys.path.insert(0, '/content/repo')

from src.classifiers import ClassifierFactory

seeds = [1, 2, 3, 4, 5]
all_results = []

cnn = ClassifierFactory.create("mobilenet_v2")
cnn.make_sub_datasets()

for s in seeds:
    save_name = f"mobilenetv2_frozen_seed_{s}.keras"
    print(f"\n--- Training seed {s} ---")

    model = cnn.train_one_run(
        seed=s,
        save_name=save_name,
        epochs=10,
        trainable_backbone=False,
        learning_rate=1e-3,
    )

    result = cnn.evaluate(model=model)
    result["seed"] = s
    result["save_name"] = save_name
    all_results.append(result)

    print(f"Seed {s}: accuracy={result['accuracy']:.4f}, macro_f1={result['macro_f1']:.4f}")

## 5. Summary

In [ ]:
accs = np.array([r["accuracy"] for r in all_results])
f1s = np.array([r["macro_f1"] for r in all_results])

summary = {
    "accuracy_mean": float(accs.mean()),
    "accuracy_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,
    "macro_f1_mean": float(f1s.mean()),
    "macro_f1_std": float(f1s.std(ddof=1)) if len(f1s) > 1 else 0.0,
}

print("=== Final Summary ===")
print(f"Accuracy: {summary['accuracy_mean']:.4f} +/- {summary['accuracy_std']:.4f}")
print(f"Macro F1: {summary['macro_f1_mean']:.4f} +/- {summary['macro_f1_std']:.4f}")

## 6. Copy trained models to Google Drive

In [ ]:
import shutil

drive_models = "/content/drive/MyDrive/feline-skin-disease-detection/trained_models"
os.makedirs(drive_models, exist_ok=True)

for f in os.listdir("trained_models"):
    if f.endswith(".keras"):
        src = os.path.join("trained_models", f)
        dst = os.path.join(drive_models, f)
        shutil.copy2(src, dst)
        print(f"Copied {f} to Drive")

print("Done! Models saved to Google Drive.")